
# Single-turn simulation (Phase 3)

Runs two-agent single-turn simulations using `SingleTurnSimulator`:
- seed agent A with a target emotion (synthetic templates aligned to DistilRoBERTa labels)
- agent B replies with a specified strategy prompt
- agent A follows up
- classify emotions before/after using DistilRoBERTa

Requires `OPENAI_API_KEY` in the environment (OpenAI Chat completions). Outputs: CSV and optional heatmap to `results/`.


In [1]:
import getpass, os
os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')


In [2]:
# Optional: install/refresh package in hosted Colab
!pip uninstall -y dynamic-conversation dynamic_conversation
!pip cache purge
!pip install --no-cache-dir git+https://github.com/Javin-Mendiratta/Dynamic-Conversation.git@derek_12_13

import os
from dynamic_conversation import SingleTurnSimulator, ResponseStrategy, compute_transition_confidence_intervals

# Ensure your OpenAI key is set. Uncomment and set directly if running interactively.
# # os.environ["OPENAI_API_KEY"] = "sk-..."


Found existing installation: dynamic-conversation 0.1.1
Uninstalling dynamic-conversation-0.1.1:
  Successfully uninstalled dynamic-conversation-0.1.1
Files removed: 12
  Cloning https://github.com/Javin-Mendiratta/Dynamic-Conversation.git (to revision derek_12_13) to /tmp/pip-req-build-fcmdl6a1
  Running command git clone --filter=blob:none --quiet https://github.com/Javin-Mendiratta/Dynamic-Conversation.git /tmp/pip-req-build-fcmdl6a1
  Running command git checkout -b derek_12_13 --track origin/derek_12_13
  Switched to a new branch 'derek_12_13'
  Branch 'derek_12_13' set up to track remote branch 'derek_12_13' from 'origin'.
  Resolved https://github.com/Javin-Mendiratta/Dynamic-Conversation.git to commit 2822f70c626e9f9baea74dd9d6952de575b09030
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dynamic-conversation: filename=dynamic_conversation-0.1.1-py3-none-any.whl size=21690

In [ ]:
# Full Phase 3 grid (baseline included). Increase runs_per_pair for real experiments.
emotions = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
strategies = [
    ResponseStrategy.VALIDATE,
    ResponseStrategy.EXPLORE,
    ResponseStrategy.REFRAME,
    ResponseStrategy.AFFIRM,
    ResponseStrategy.GUIDE,
    ResponseStrategy.NORMALIZE,
]

runs_per_pair = 1  # TODO: raise to 15-20 for report-ready stats
style_modifier = None  # e.g., 'empathetic and casual'
use_llm_seed = False
include_baseline = True

sim = SingleTurnSimulator(use_gpu=True)
df = sim.run_batch(
    emotions=emotions,
    strategies=strategies,
    runs_per_pair=runs_per_pair,
    style_modifier=style_modifier,
    use_llm_seed=use_llm_seed,
    include_baseline=include_baseline,
    save_csv='results/full_phase3_single_turn.csv',
    save_heatmap='results/full_phase3_single_turn_heatmap.png',
)

df.head()


Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


AttributeError: 'str' object has no attribute 'with_suffix'


Default grid runs all 7 emotions × 6 strategies + baseline. Adjust `runs_per_pair`, `style_modifier`, `use_llm_seed`, and output paths as needed. Keep outputs under `results/` and leave `save_metadata=True` (default) to log configs.



Use `compute_transition_confidence_intervals` to summarize shifts with CIs. For comparative analyses (e.g., strategy vs. baseline), rerun with higher `runs_per_pair` (15–20) and, if possible, another seed/style modifier.


In [ ]:
# Compute Wilson CIs for emotion distributions per (intended_emotion, strategy)
ci_df = compute_transition_confidence_intervals(df, confidence=0.95)
ci_df.head()


In [ ]:
# Inspect saved run metadata (config/seeds)
import json
from pathlib import Path
meta_path = Path("results/demo_single_turn.meta.json")
if meta_path.exists():
    print(meta_path.read_text())
else:
    print("Metadata file not found; ensure save_metadata=True in run_batch.")
